# Embedding Experiments — Stratified Permutation Tests

Four analyses using `stratified_permutation_test` from `orgpackage/tester.py`:

| # | Analysis | Factor | Stratification |
|---|----------|--------|----------------|
| 1 | **Shot count** | 0 vs 1 vs Few-shot | Model |
| 2 | **Model similarity** | Model comparison | Domain |
| 3 | **Classifier head** | LR vs SVM | Model |
| 4 | **Model classifier** | Model comparison | Domain |

Results cached in `results/correctness_tables/`. Global **Holm correction** applied.

## 0. Imports & Setup

In [ ]:
import os, sys, ast, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity
from statsmodels.stats.multitest import multipletests

project_root = os.path.abspath(".")
if project_root not in sys.path: sys.path.insert(0, project_root)
os.chdir(project_root)

from orgpackage.aux import load_experiments, load_trained_model, load_dataset
from orgpackage.config import DOMAIN_CLASSES_CORR
from orgpackage.tester import stratified_permutation_test

warnings.filterwarnings("ignore")
CORRECTNESS_DIR = "./results/correctness_tables"
os.makedirs(CORRECTNESS_DIR, exist_ok=True)
DOMAINS = ["medical", "administrative", "education"]
ALL_LABELS = ["hospital", "university_hospital", "local_government", "primary_school", "secondary_school"]
print("Setup complete.")

## 1. Load Data

In [ ]:
exps = load_experiments()
emb_exps = exps[exps["Technique"] == "embedding"]
sim_exps = emb_exps[emb_exps["Method"] == "similarity"]
clf_exps = emb_exps[emb_exps["Method"] == "classifier"]

print("Loading master dataset…")
dataset = load_dataset()
dataset["instance"] = dataset["instance"].str.strip().str.lower().str.replace('https://', 'http://')
for c in ALL_LABELS: dataset[c] = dataset[c].fillna(0).astype(int) if c in dataset.columns else 0
print(f"{len(dataset)} items loaded.")

## 2. Helpers

In [ ]:
def _normalize_uri(uri):
    if not isinstance(uri, str): return uri
    return uri.strip().lower().replace('https://', 'http://')

def _normalize_path(p):
    if not isinstance(p, str): return p
    p = p.replace('\\', '/').replace('./', '').replace('.\/', '')
    if os.path.exists(p): return p
    fallback = os.path.join("results/trained_models", os.path.basename(p))
    return fallback if os.path.exists(fallback) else p

def _load_emb(model_name):
    path = f"results/embeddings/{model_name}_embeddings.csv"
    if not os.path.exists(path): raise FileNotFoundError(path)
    df = pd.read_csv(path, low_memory=False); col = f"{model_name}_embedding"
    
    def _parse(v):
        if pd.isna(v): return None
        try: return np.array(ast.literal_eval(str(v).replace('np.array(', '[').replace(')', ']')))
        except: return None
        
    df[col] = df[col].apply(_parse)
    df = df[df[col].apply(lambda x: isinstance(x, np.ndarray) and np.issubdtype(x.dtype, np.number) and not np.isnan(x).any())].copy()
    if df.empty: return pd.DataFrame(), col
    df["instance"] = df["instance"].apply(_normalize_uri)
    merged = df.merge(dataset[["instance"] + ALL_LABELS], on="instance", how="inner")
    return merged, col

def _pred_sim(emb, protos, threshold, multiclass, classes):
    x = emb.reshape(1, -1); sims = {c: -1.0 for c in classes}
    for c, p in protos.items():
        best = -1.0; plist = p.values() if isinstance(p, dict) else [p]
        for v in plist:
            if v is None: continue
            v = np.array(v)
            if not np.isnan(v).any(): best = max(best, cosine_similarity(x, v.reshape(1,-1))[0][0])
        sims[c] = best
    preds = {c: 0 for c in classes}
    if multiclass:
        for c in classes: 
            if sims[c] >= threshold: preds[c] = 1
    else:
        bc = max(sims, key=sims.get)
        if sims[bc] >= threshold: preds[bc] = 1
    return preds

## 3. Build Tables

In [ ]:
def build_sim_table(m, ids, exps_df):
    df, col = _load_emb(m)
    if df.empty: return pd.DataFrame()
    res = {}
    for eid in ids:
        row = exps_df[exps_df.ID == eid].iloc[0]; p = row.Parameters
        cl = DOMAIN_CLASSES_CORR[row.Domain]; th = 1.0 - p.distance
        mc = p.structure == "2-multiclass"; hits = []
        for _, r in df.iterrows():
            pr = _pred_sim(r[col], p.prototypes, th, mc, cl)
            hits.append(int(all(pr[c] == int(r[c]) for c in cl)))
        res[eid] = pd.Series(hits, index=df.instance)
    return pd.DataFrame(res).astype(int)

def build_clf_table(m, ids, exps_df):
    df, col = _load_emb(m)
    if df.empty: return pd.DataFrame()
    X = np.stack(df[col].values); res = {}
    for eid in ids:
        row = exps_df[exps_df.ID == eid].iloc[0]; p = row.Parameters; cl = DOMAIN_CLASSES_CORR[row.Domain]
        st = p.get("structure"); pts = [_normalize_path(p.get("trained_classifier_hospital")), _normalize_path(p.get("trained_classifier_university_hospital"))] if st == "nested-class" else [_normalize_path(p.get("trained_classifier"))]
        if not all(pt and os.path.exists(pt) for pt in pts): continue
        try:
            if st == "nested-class":
                yh = load_trained_model(pts[0]).predict(X); yu = np.zeros_like(yh)
                pos = np.where(yh == 1)[0]
                if len(pos): yu[pos] = load_trained_model(pts[1]).predict(X[pos])
                corr = ((yh == df.hospital) & (yu == df.university_hospital)).astype(int)
            else:
                clf = load_trained_model(pts[0]); raw = clf.predict(X); yt = df[cl].values
                if yt.shape[1] > 1: 
                    yp = np.zeros_like(yt)
                    if raw.ndim == 1: 
                        for i, v in enumerate(raw): 
                            if v < yt.shape[1]: yp[i, v] = 1
                    else: yp = raw.astype(int)
                else: yp = raw.reshape(-1, 1).astype(int)
                corr = (yp == yt).all(axis=1).astype(int)
            res[eid] = pd.Series(corr.values if hasattr(corr, 'values') else corr, index=df.instance)
        except Exception as e: print(f"Error {eid}: {e}")
    return pd.DataFrame(res).astype(int)

def _get_id(index, m, k, d):
    for eid in index.get(m, {}).get(k, []):
        if eid.startswith(d[:3]): return eid
    return None

sim_index = {}; clf_index = {}
for _, r in sim_exps.iterrows(): sim_index.setdefault(r.Parameters['model'], {}).setdefault(r.Parameters['n_shot'], []).append(r.ID)
for _, r in clf_exps.iterrows(): 
    p = r.Parameters; t = "lr" if "solver" in p else ("svm" if "kernel" in p else "unknown")
    clf_index.setdefault(r.Parameters['model'], {}).setdefault(t, []).append(r.ID)

sim_ct = {}; clf_ct = {}
for m in sim_index:
    c = os.path.join(CORRECTNESS_DIR, f"{m}_sim.csv")
    if os.path.exists(c): sim_ct[m] = pd.read_csv(c, index_col=0)
    else: 
        ids = [i for s in sim_index[m].values() for i in s]; ct = build_sim_table(m, ids, sim_exps)
        if not ct.empty: sim_ct[m] = ct; ct.to_csv(c)
for m in clf_index:
    c = os.path.join(CORRECTNESS_DIR, f"{m}_clf.csv")
    if os.path.exists(c): clf_ct[m] = pd.read_csv(c, index_col=0)
    else:
        ids = [i for t in clf_index[m].values() for i in t]; ct = build_clf_table(m, ids, clf_exps)
        if not ct.empty: clf_ct[m] = ct; ct.to_csv(c)

## 4. Tests & Plots

In [ ]:
all_res = []
def run_p_test(strata, lbl, tag):
    if not strata: return pd.DataFrame()
    r = stratified_permutation_test(strata, "A", "B", 10000, "pooled")
    return pd.DataFrame([{"analysis": tag, "comparison": lbl, "obs_diff": r[0], "p_value": r[1]}])

# A1: Shots
for sa, sb, lbl in [("1_shot", "0_shot", "1-shot vs 0-shot"), ("few_shot", "1_shot", "few vs 1-shot"), ("few_shot", "0_shot", "few vs 0-shot")]:
    st = []
    for m, ct in sim_ct.items():
        dfm = []
        for d in DOMAINS:
            ia, ib = _get_id(sim_index, m, sa, d), _get_id(sim_index, m, sb, d)
            if ia in ct.columns and ib in ct.columns: dfm.append(ct[[ia, ib]].rename(columns={ia: "A", ib: "B"}))
        if dfm: st.append(pd.concat(dfm))
    all_res.append(run_p_test(st, lbl, "A1_shot"))

# A2: Sim Models
for sa in ["0_shot", "1_shot", "few_shot"]:
    mdls = sorted(sim_ct.keys())
    for ma, mb in combinations(mdls, 2):
        st = []
        for d in DOMAINS:
            ia, ib = _get_id(sim_index, ma, sa, d), _get_id(sim_index, mb, sa, d)
            if ma in sim_ct and mb in sim_ct and ia in sim_ct[ma].columns and ib in sim_ct[mb].columns:
                jo = sim_ct[ma][[ia]].join(sim_ct[mb][[ib]], how="inner").rename(columns={ia: "A", ib: "B"})
                if not jo.empty: st.append(jo)
        all_res.append(run_p_test(st, f"{ma} vs {mb}", f"A2_model_{sa}"))

# A3: LR vs SVM
st = []
for m, ct in clf_ct.items():
    lr = [_get_id(clf_index, m, "lr", d) for d in DOMAINS]; sv = [_get_id(clf_index, m, "svm", d) for d in DOMAINS]
    ia = [i for i in lr if i in ct.columns]; ib = [i for i in sv if i in ct.columns]
    if ia and ib: st.append(pd.concat([ct[ia].mean(axis=1), ct[ib].mean(axis=1)], axis=1).rename(columns={0: "A", 1: "B"}))
all_res.append(run_p_test(st, "LR vs SVM", "A3_clf_head"))

# A4: Clf Models
for head in ["lr", "svm"]:
    mdls = sorted(clf_ct.keys())
    for ma, mb in combinations(mdls, 2):
        st = []
        for d in DOMAINS:
            ia, ib = _get_id(clf_index, ma, head, d), _get_id(clf_index, mb, head, d)
            if ma in clf_ct and mb in clf_ct and ia in clf_ct[ma].columns and ib in clf_ct[mb].columns:
                jo = clf_ct[ma][[ia]].join(clf_ct[mb][[ib]], how="inner").rename(columns={ia: "A", ib: "B"})
                if not jo.empty: st.append(jo)
        all_res.append(run_p_test(st, f"{ma} vs {mb}", f"A4_model_{head}"))

## 5. Summary

In [ ]:
final = pd.concat([r for r in all_res if not r.empty], ignore_index=True)
rej, pc, _, _ = multipletests(final.p_value, method="holm")
final["p_corrected"] = pc; final["significant"] = rej
final.to_csv("results/embedding_permutation_tests.csv", index=False)

sns.set_theme(style="whitegrid", palette="pastel")
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
for i, t in enumerate(["A1_shot", "A2_model", "A3_clf_head", "A4_model"]):
    sub = final[final.analysis.str.startswith(t)]
    if sub.empty: continue
    clrs = ["#2ecc71" if (s and d > 0) else ("#e74c3c" if s else "#95a5a6") for s, d in zip(sub.significant, sub.obs_diff)]
    sns.barplot(data=sub, x="comparison", y="obs_diff", ax=axes[i], palette=clrs)
    axes[i].set_title(t, fontweight='bold'); axes[i].tick_params(axis='x', rotation=45)
    for p, bar in zip(sub.p_corrected, axes[i].patches):
        if p < 0.05: axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height(), '*', ha='center')
plt.tight_layout(); fig.savefig("figures/perm_tests.png", dpi=300); plt.show()
display(final)